![AES_744 Masthead](https://kyrill.ias.sdsmt.edu/wjc/eduresources/AES_519_Masthead.png)
# Fast Tutorial: Reading RData Files in Python

## Introduction

The R Data File Format (often identified by the suffix *.RData or a similar filename extension) is a binary data format designed to work with the R Statistical Package.  It is designed to hold scalar, tabular and array data in a compressed form for fast storage and access.

Python doesn't have a native way to read this data easily, so there are packages designed to crack these files open.

This tutorial focuses on using one package: [Otto Fajardo's "PyReadR" package](https://github.com/ofajardo/pyreadr)

This package permits the ingestion of RData files into pandas, and specifically facilitates their loading into the [Pandas](https://pandas.pydata.org/) data model, which is Python's "go-to" data model for working with tabular data.  It also permits loading data locally from a file on your machine's file system or from files stored remotely on HTTP(S) services.

This tutorial will include examples of both. For one example, you will be expected to download an example file locally to your computer.


---
## Libraries

For this exercise, you will be using the following Python Libraries.   Not all are essential to downloading and accessing RData Files but they will help in displaying them for teaching purposes.


### The Basics...

* [numpy](https://numpy.org/): Python's "go-to" library for math and array/matrix structures. Always good to have handy for crunching whether you will acutally use it explicitly or not.
* [pandas](https://pandas.pydata.org/): A fast, powerful, flexible and easy to use open source data analysis and manipulation tool that has structures similar to R's "DataFrame"
* [pyreadr](https://github.com/ofajardo/pyreadr): Otto Fajardo's "PyReadR" package.  Sadly the examples are on the GitHub landing page rather than a fancy "readthedocs."  That said, it's slim and it works!

For accessing data stored off-site on an HTTP(S) service, you also want the following.  These come with Python (as opposed to Pandas and Numpy but you will need to call them).


* [io](https://docs.python.org/3/library/io.html): This is part of Python's core library set and is used in input and output.
* [socket](https://docs.python.org/3/library/socket.html): Socket is the internal core library set that manages basic networking connections.  To go off your machine, you will need this to get out the door.
* [urllib.request](https://docs.python.org/3/library/urllib.request.html#module-urllib.request): To get into *somebody else's* door, you will be using the core urllib library and specifically the "request" sublibrary to query specific URL web addresses, like the RData File that we will be accessing in this exercise.

### Fancy Tools

You won't need these directly, but they are rather nice to have.  Some of these only work when inside an interactive graphical IPython (e.g., Jupyter) environment.  If you need to convert this into a simple Python script, you will need to remove calls to these libraries.

* [itables](https://mwouts.github.io/itables/): Marc Wouts's Jupyter extension that will allow your tables to be displayed in a manner similar to RStudio, by which you can scroll through them.  If you have big or wide tables, this is what you need to explore your data.  Be aware that you should comment it out if you export your file into a plain *.py executable.
  
To use ITables on demand, you can use the function <a href="https://mwouts.github.io/itables/apps/notebook.html">itables.show()</a></p>
</div>

### Non-Library IPython Resource...

* [version_information](https://github.com/jrjohansson/version_information): Package version documentation to aid in replicability. See its use at the bottom of this notebook.

See its use at the bottom of this notebook.  I like to use it to list all the packages I use in a given notebook and log their versions, as well as the operating system I'm using for a given exercise.  You will need to remove this before saving it as a plain Python "*.py" script.


In [1]:
######################################################
#
# Libraries
#

import numpy              as np
import pandas             as pd

import pyreadr

import io           
import socket           
import urllib.request        

import itables

#
######################################################

---
## Our Data File 

For this exercise, I have a simple single R "DataFrame" (they now call them "tibbles") inside an RData file.  It's a simple lookup table for the US Climate Divisions.

You can download the file from here:

[https://thredds.ias.sdsmt.edu:8443/thredds/fileServer/LOCA2/Specific_Regional_Aggregate_Sets/NCEI_Climate_Divisions/NCEI_nClimDiv_LUT.RData](https://thredds.ias.sdsmt.edu:8443/thredds/fileServer/LOCA2/Specific_Regional_Aggregate_Sets/NCEI_Climate_Divisions/NCEI_nClimDiv_LUT.RData)

The link will allow you to directly download the file, and later you will see how to access the file directly from the remote HTTPS services we maintain here at South Dakota Mines.

The first example, however, will have us read the file locally.  The script below simply has it read from my download directory.  You may need to change it for your machine.

---
## Opening and Reading a Local RData File.

Reading a file locally is very easy and on par with reading a comma-delimited file (CSV File) or an Excel spreadsheet, but with one extra step.

Here we will open the file using [pyreadr.read_r()](https://github.com/ofajardo/pyreadr#basic-usage--reading-files).  This will access all of the available data variables, tables, etc., in the file.  The results will appear as a Python "ordered dictionary."

In [2]:
######################################################
#
# Cracking a local RData file
#

# Local File Path

local_RData_file     = "/Users/wjc/Downloads/NCEI_nClimDiv_LUT.RData"

local_RData_contents = pyreadr.read_r(path = local_RData_file)

local_RData_contents

#
######################################################

OrderedDict([('NCEI_nClimDiv_LUT',
                  climdiv climdiv_state_code cdnew state_fips fips_cd  \
              0      0101                 01    01         01    0101   
              1      0102                 01    02         01    0102   
              2      0103                 01    03         01    0103   
              3      0104                 01    04         01    0104   
              4      0105                 01    05         01    0105   
              ..      ...                ...   ...        ...     ...   
              339    4806                 48    06         56    5606   
              340    4807                 48    07         56    5607   
              341    4808                 48    08         56    5608   
              342    4809                 48    09         56    5609   
              343    4810                 48    10         56    5610   
              
                                   climdiv_name climdiv_state_abrv climdiv

As you can see, it's pretty messy, but it also assumes you can include more than one data table, array, variable, etc. If you want to inspect just the list data fields that are kept in the RData file, you can just query the file's "keys" just like you would with a Python dictionary.

In [3]:
######################################################
#
# Peruse the inventory of the RData File.
#

local_RData_contents.keys()

#
######################################################

odict_keys(['NCEI_nClimDiv_LUT'])

Accessing the file as [pandas.DataTable](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) can then be pulled by accessing the contents via the DataFrame's key.  In this case, our key is called "NCEI_nClimDiv_LUT".

Here we are also accessing the file in a "pretty" and browsable format with [iTables.show()](https://itables.org/apps/notebook.html#show).

(Note that the default for displaying a DataFrame in iTables is to suppress unnamed "indices" columns. I am using arguments to address that here.)

In [4]:
######################################################
#
# Extract and Display the NCEI Lookup Table.
#

local_df = local_RData_contents['NCEI_nClimDiv_LUT']

itables.show(df        = local_df, 
             showIndex =     True)

#
######################################################

Loading ITables v2.8.1 from the internet... (need help?)


And with that, you are ready to use Pandas or GeoPandas as normal.

---
## Remote Access of an RData File Kept on an HTTP Service.

There are a few more steps to access the same file remotely from an HTTP service.  

The steps are presented here in slow motion, but as you can see below, I only have one extra line before invoking *pyreadr.read_r()*.

* Provide a remote URL to the file location on your HTTP service.
* Create a request to access that remote path with [urllib.request.urlopen()](https://docs.python.org/3/library/urllib.request.html#urllib.request.urlopen)
* Read the raw ones-and-zeros from the URL (that's the ".read()")
* send that stream with [io.BytesIO()](https://docs.python.org/3/library/io.html#io.BytesIO) into *pyreadr.readr()*.
* ... and from there, just pull the dataframe with the proper key as before.


In [5]:
######################################################
#
# Remote Access of an RData File Kept on an HTTP Service.
#

remote_RData_Path = "https://thredds.ias.sdsmt.edu:8443/thredds/fileServer/"         + \
                    "LOCA2/Specific_Regional_Aggregate_Sets/NCEI_Climate_Divisions/" + \
                    "NCEI_nClimDiv_LUT.RData"

response              = urllib.request.urlopen(remote_RData_Path)
remote_RData_contents = pyreadr.read_r(io.BytesIO(response.read()))

remote_df = remote_RData_contents['NCEI_nClimDiv_LUT']

itables.show(df        = remote_df, 
             showIndex =      True)

#
######################################################

Loading ITables v2.8.1 from the internet... (need help?)


---
## Version Information

*Version Information* is a package developed by J.R. Johansson and operates as what the iPython/Jupyter Environments call ["Magic Line" commands](https://jakevdp.github.io/PythonDataScienceHandbook/01.03-magic-commands.html), which are invoked by a "%."  This one is convenient since it displays detailed information on the versions of Python, Jupyter, and any other denoted packages (but for those, you DO need to delete them since this is running "outside" of your notebook program.

The output will be in JSON format in the Jupyter Lab Notebook, but it will be a professional-looking table when rendered as an HTML file.

In [6]:
################################################################
#
# Loading Version Information
#

%load_ext version_information
%version_information numpy, pandas, pyreadr, itables, version_information

#
################################################################

Software versions
Python 3.12.12 64bit [Clang 19.1.7 ]
IPython 9.14.1
OS macOS 26.5.2 arm64 arm 64bit
numpy 2.4.6
pandas 3.0.3
pyreadr 0.5.6
itables 2.8.1
version_information 1.0.4
Thu Jul 23 14:03:13 2026 MDT